In [ ]:
import requests
from bs4 import BeautifulSoup
import re # Dùng thư viện Regular Expression để tìm class

url = "https://bonbanh.com/oto"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

res = requests.get(url, headers=headers)
res.encoding = 'utf-8'
soup = BeautifulSoup(res.text, 'html.parser')

# 1. Tìm tất cả các thẻ <li> có class chứa chữ 'car-item' (bỏ qua khác biệt giữa row1, row2)
car_items = soup.find_all('li', class_=re.compile('car-item'))

for item in car_items[:5]: # In thử 5 chiếc đầu tiên
    # 2. Lấy thẻ <a> bên trong
    a_tag = item.find('a', itemprop='url')
    
    if a_tag:
        # 3. Trích xuất thông tin từ thuộc tính của thẻ <a>
        chuoi_thong_tin = a_tag.get('title')
        link_chi_tiet = "https://bonbanh.com/" + a_tag.get('href')
        
        print(f"Data thô: {chuoi_thong_tin}")
        print(f"Link: {link_chi_tiet}")
        print("-" * 50)

Data thô: Ban xe oto cu Mercedes Benz E class 2021 E200 Exclusive gia 1 Tỷ 278 Triệu - TP HCM
Link: https://bonbanh.com/xe-mercedes_benz-e_class-e200-exclusive-2021-6742414
--------------------------------------------------
Data thô: Ban xe oto VinFast VF5 2026 Plus gia 455 Triệu - Hà Nội
Link: https://bonbanh.com/xe-vinfast-vf5-plus-2026-6692319
--------------------------------------------------
Data thô: Ban xe oto cu LandRover Range Rover 2020 SVAutobiography LWB 3.0 I6 gia 7 Tỷ 900 Triệu - Hà Nội
Link: https://bonbanh.com/xe-landrover-range_rover-svautobiography-lwb-3.0-i6-2020-6688745
--------------------------------------------------
Data thô: Ban xe oto cu Porsche Panamera 2021 4 Executive gia 5 Tỷ 150 Triệu - Hà Nội
Link: https://bonbanh.com/xe-porsche-panamera-4-executive-2021-6679921
--------------------------------------------------
Data thô: Ban xe oto cu Mercedes Benz G class 2020 G63 AMG gia 6 Tỷ 886 Triệu - Hà Nội
Link: https://bonbanh.com/xe-mercedes_benz-g_class-g63-am

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def clean_price(price_str):
    """Hàm chuyển đổi chuỗi giá thành con số (Đơn vị: Triệu VNĐ)"""
    price_str = price_str.lower().replace(" ", "")
    ty = 0
    trieu = 0
    
    try:
        if 'tỷ' in price_str or 'ty' in price_str:
            parts = re.split(r'tỷ|ty', price_str)
            ty = int(re.sub(r'\D', '', parts[0])) if parts[0] else 0
            trieu = int(re.sub(r'\D', '', parts[1])) if len(parts) > 1 and parts[1] else 0
        else:
            trieu = int(re.sub(r'\D', '', price_str)) if price_str else 0
        return ty * 1000 + trieu
    except:
        return None # Trả về None nếu gặp lỗi (ví dụ: "Giá thỏa thuận")

def parse_car_title(raw_title):
    """Hàm 'chẻ' chuỗi tiêu đề thành Hãng/Mẫu, Năm SX, Giá, Địa điểm"""
    # 1. Bỏ đi các tiền tố không cần thiết
    title = raw_title.replace("Ban xe oto cu ", "").replace("Ban xe oto ", "").strip()
    
    # 2. Tìm Giá và Địa điểm (Nằm sau chữ 'gia' và chia cắt bởi dấu '-')
    price_match = re.search(r'gia\s+(.*?)\s+-\s+(.*)', title, re.IGNORECASE)
    gia_tien = None
    dia_diem = None
    
    if price_match:
        gia_tien_str = price_match.group(1).strip()
        dia_diem = price_match.group(2).strip()
        gia_tien = clean_price(gia_tien_str)
        # Cắt bỏ phần giá và địa điểm khỏi chuỗi để tìm tên xe và năm
        title = title[:price_match.start()].strip() 
        
    # 3. Tìm Năm Sản Xuất (4 chữ số liên tiếp, ví dụ: 2018, 2021)
    year_match = re.search(r'\b(19\d{2}|20\d{2})\b', title)
    nam_sx = None
    hang_mau = title # Mặc định phần còn lại là tên xe
    
    if year_match:
        nam_sx = int(year_match.group(1))
        # Nếu muốn tên xe sạch hơn, bạn có thể chỉ lấy phần trước năm sản xuất
        # Ví dụ: "Mercedes Benz E class" thay vì kèm cả "E200 Exclusive" ở sau
        hang_mau = title[:year_match.start()].strip()

    return {
        "Hãng_Mẫu": hang_mau,
        "Năm_SX": nam_sx,
        "Giá_Triệu": gia_tien,
        "Địa_Điểm": dia_diem,
        "Chuỗi_Gốc": raw_title # Lưu lại để đối chiếu nếu cần
    }

def scrape_bonbanh_mass(pages=5):
    """Hàm cào diện rộng qua nhiều trang"""
    base_url = "https://bonbanh.com/oto"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    
    all_cars = []
    
    for page in range(1, pages + 1):
        print(f"Đang cào trang {page}...")
        url = f"{base_url}/page,{page}"
        
        try:
            res = requests.get(url, headers=headers, timeout=10)
            res.encoding = 'utf-8'
            soup = BeautifulSoup(res.text, 'html.parser')
            
            # Tìm tất cả các thẻ li có class chứa 'car-item'
            items = soup.find_all('li', class_=re.compile('car-item'))
            
            for item in items:
                a_tag = item.find('a', itemprop='url')
                if a_tag and a_tag.get('title'):
                    raw_title = a_tag.get('title')
                    parsed_data = parse_car_title(raw_title)
                    all_cars.append(parsed_data)
                    
            # Dừng 1 chút để không bị server Bonbanh chặn IP
            time.sleep(1)
            
        except Exception as e:
            print(f"Lỗi ở trang {page}: {e}")
            continue
            
    return pd.DataFrame(all_cars)

# BẮT ĐẦU CHẠY THỰC TẾ
# Bạn có thể tăng pages=50 hoặc 100 để lấy lượng lớn dữ liệu
df_cars = scrape_bonbanh_mass(pages=3) 

# In ra 5 dòng đầu tiên xem thành quả


# Lưu thẳng ra file CSV để làm EDA và Train Model
df_cars.to_csv("du_lieu_xe_cu.csv", index=False, encoding='utf-8-sig')
print("\nĐã lưu thành công file du_lieu_xe_cu.csv!")

Đang cào trang 1...
Đang cào trang 2...
Đang cào trang 3...
                 Hãng_Mẫu  Năm_SX  Giá_Triệu   Địa_Điểm  \
0   Mercedes Benz E class    2021       1278     TP HCM   
1             VinFast VF5    2026        455     Hà Nội   
2   LandRover Range Rover    2020       7900     Hà Nội   
3        Porsche Panamera    2021       5150     Hà Nội   
4   Mercedes Benz G class    2020       6886     Hà Nội   
5           Toyota Innova    2025        815     Hà Nội   
6                  BMW X7    2025       5999     Hà Nội   
7            Ford Transit    2022        515     TP HCM   
8      Chevrolet Colorado    2016        365     TP HCM   
9           Porsche Macan    2021       2588     TP HCM   
10      Mercedes Benz GLC    2021       1460  Hải Phòng   
11                Peugeot    2008        575  Hải Phòng   
12              Mazda CX5    2024        745  Hải Phòng   
13                Mazda 6    2022        665  Hải Phòng   
14                  MG ZS    2025        535  Hải Phòng

In [ ]:
print(df_cars.head(60))

In [7]:
import os
duong_dan = os.path.abspath("du_lieu_xe_cu.csv")
print(f"File CSV của bạn đang nằm ở đây nè: {duong_dan}")

File CSV của bạn đang nằm ở đây nè: c:\Users\DELL\AppData\Local\Programs\Microsoft VS Code\du_lieu_xe_cu.csv
